In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO

import numpy as np

#Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.express as px

import time
from datetime import date
import matplotlib.pyplot as plt

In [2]:
def get_ipos() -> pd.DataFrame:
    """
    Fetch IPO data for the given year from stockanalysis.com.
    """
    url = f"https://www.iposcoop.com/2025-pricings/"
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/58.0.3029.110 Safari/537.3'
        )
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        # Wrap HTML text in StringIO to avoid deprecation warning
        # "Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object."
        html_io = StringIO(response.text)
        tables = pd.read_html(html_io)

        if not tables:
            raise ValueError(f"No tables found")

        return tables[0]

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
    except ValueError as ve:
        print(f"Data error: {ve}")
    except Exception as ex:
        print(f"Unexpected error: {ex}")

    return pd.DataFrame()

In [3]:
raw_df = get_ipos()

In [4]:
df = raw_df.copy()

In [5]:
df["Offer Date"] = pd.to_datetime(df["Offer Date"], errors="coerce")

In [6]:
offer_date_filter = (df["Offer Date"] < pd.to_datetime("2025-09-01"))

In [7]:
filtered_df = df[offer_date_filter].copy()

In [8]:
filtered_df2 = filtered_df.copy()

In [9]:
filtered_df2 = filtered_df2.rename(columns={"Return": "Return (Percentage)"})

In [10]:
filtered_df2["Return (Percentage)"] = pd.to_numeric(filtered_df2["Return (Percentage)"].str.replace("%", "", regex=False),
    errors="coerce")

In [11]:
filtered_df2.columns

Index(['Company', 'Symbol', 'Industry', 'Offer Date', 'Shares (millions)',
       'Offer Price', '1st Day Close', 'Current Price', 'Return (Percentage)',
       'SCOOP Rating'],
      dtype='object')

In [13]:
filter_returns = ~(filtered_df2["Return (Percentage)"] == 0)

In [14]:
df2 = filtered_df2.copy()

In [15]:
df2 = filtered_df2[filter_returns]

In [16]:
df2

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return (Percentage),SCOOP Rating
68,"Picard Medical, Inc.",PMI,Health Care,2025-08-29,4.3,$4.00,$4.59,$1.68,-58.00,S/O
69,GrowHub Limited (The),TGHL,Technology,2025-08-28,3.7,$4.00,$3.38,$0.40,-90.00,S/O
70,TryHard Holdings Limited,THH,Consumer Services,2025-08-28,1.5,$4.00,$6.02,$29.47,636.75,S/O
71,Curanex Pharmaceuticals,CURX,Health Care,2025-08-26,3.8,$4.00,$4.02,$0.37,-90.75,S/O
72,"Cantor Equity Partners IV, Inc. (Stock-Only SPAC)",CEPF,Blank Check,2025-08-21,40.0,$10.00,$10.12,$10.12,1.20,S/O
...,...,...,...,...,...,...,...,...,...,...
226,Mint Inc. Ltd.,MIMI,Other,2025-01-10,1.8,$4.00,$4.11,$5.30,32.50,S/O
227,3 E Network Technology Group Limited,MASK,Technology,2025-01-08,1.3,$4.00,$3.25,$2.75,-31.25,S/O
228,"Cantor Equity Partners I, Inc. (Stock-Only SPAC)",CEPO,Blank Check,2025-01-07,20.0,$10.00,$10.05,$10.10,1.00,S/O
229,Zhengye Biotechnology Holding Limited,ZYBT,Health Care,2025-01-07,1.5,$4.00,$4.86,$4.22,5.50,S/O


In [17]:
#RESET stocks_df to original copy

In [18]:
stocks_df = df2.copy()

In [19]:
df3 = list(df2["Symbol"])

In [20]:
stocks = []
for ticker_name in df3:
	ticker_obj = yf.Ticker(ticker_name)
	stocks.append(ticker_obj)

In [53]:
history_list = []

for item in stocks:
    item_history = item.history(
        start="2025-01-01",
        end="2026-09-30",
        interval="1d"
    )

    if not item_history.empty:
        item_history = item_history.reset_index()
        item_history["Ticker"] = item.ticker

        history_list.append(item_history)

concat_df = pd.concat(
    history_list,
    ignore_index=True
)

$MJID: possibly delisted; no timezone found
$EMPG: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found
$TBH: possibly delisted; no timezone found
$WGRX: possibly delisted; no timezone found
$AGH: possibly delisted; no timezone found
$EPWK: possibly delisted; no timezone found
$MTSR: possibly delisted; no timezone found
$SKBL: possibly delisted; no timezone found
$MCTR: possibly delisted; no timezone found


In [54]:
main_df = concat_df.copy()

In [55]:
main_df

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Ticker,Capital Gains
0,2025-08-29 00:00:00-04:00,210.50,356.00,209.500,229.50,126794,0.0,0.0,PMI,NaN
1,2025-09-02 00:00:00-04:00,242.50,267.50,232.500,257.00,98358,0.0,0.0,PMI,NaN
2,2025-09-03 00:00:00-04:00,257.00,270.00,250.500,264.50,35960,0.0,0.0,PMI,NaN
3,2025-09-04 00:00:00-04:00,265.50,287.50,265.000,286.50,28074,0.0,0.0,PMI,NaN
4,2025-09-05 00:00:00-04:00,288.00,295.00,282.500,292.50,18572,0.0,0.0,PMI,NaN
...,...,...,...,...,...,...,...,...,...,...
46268,2026-09-22 00:00:00-04:00,2.65,2.95,2.650,2.80,40400,0.0,0.0,INLF,NaN
46269,2026-09-23 00:00:00-04:00,2.86,2.91,2.834,2.91,16300,0.0,0.0,INLF,NaN
46270,2026-09-24 00:00:00-04:00,2.90,3.00,2.890,2.95,9533500,0.0,0.0,INLF,NaN
46271,2026-09-25 00:00:00-04:00,6.25,6.87,4.570,5.10,49411900,0.0,0.0,INLF,NaN


In [56]:
main_df.dtypes

Date             datetime64[ns, America/New_York]
Open                                      float64
High                                      float64
Low                                       float64
Close                                     float64
Volume                                      int64
Dividends                                 float64
Stock Splits                              float64
Ticker                                     object
Capital Gains                             float64
dtype: object

In [57]:
main_df['volatility'] = main_df['Close'].rolling(30).std() * np.sqrt(252)

In [58]:
main_df.drop(
    columns=["Dividends", "Stock Splits", "Capital Gains"],
    inplace=True
)

In [59]:
main_df

,Date,Open,High,Low,Close,Volume,Ticker,volatility
0,2025-08-29 00:00:00-04:00,210.50,356.00,209.500,229.50,126794,PMI,NaN
1,2025-09-02 00:00:00-04:00,242.50,267.50,232.500,257.00,98358,PMI,NaN
2,2025-09-03 00:00:00-04:00,257.00,270.00,250.500,264.50,35960,PMI,NaN
3,2025-09-04 00:00:00-04:00,265.50,287.50,265.000,286.50,28074,PMI,NaN
4,2025-09-05 00:00:00-04:00,288.00,295.00,282.500,292.50,18572,PMI,NaN
...,...,...,...,...,...,...,...,...
46268,2026-09-22 00:00:00-04:00,2.65,2.95,2.650,2.80,40400,INLF,12.137702
46269,2026-09-23 00:00:00-04:00,2.86,2.91,2.834,2.91,16300,INLF,12.083137
46270,2026-09-24 00:00:00-04:00,2.90,3.00,2.890,2.95,9533500,INLF,11.880555
46271,2026-09-25 00:00:00-04:00,6.25,6.87,4.570,5.10,49411900,INLF,12.155549


In [60]:
main_df["growth_252d"] = (main_df["Close"] / main_df["Close"].shift(252))

In [62]:
main_df['Sharpe'] = (main_df['growth_252d'] - 0.05) / main_df['volatility']

In [63]:
main_df

,Date,Open,High,Low,Close,Volume,Ticker,volatility,growth_252d,Sharpe
0,2025-08-29 00:00:00-04:00,210.50,356.00,209.500,229.50,126794,PMI,NaN,NaN,NaN
1,2025-09-02 00:00:00-04:00,242.50,267.50,232.500,257.00,98358,PMI,NaN,NaN,NaN
2,2025-09-03 00:00:00-04:00,257.00,270.00,250.500,264.50,35960,PMI,NaN,NaN,NaN
3,2025-09-04 00:00:00-04:00,265.50,287.50,265.000,286.50,28074,PMI,NaN,NaN,NaN
4,2025-09-05 00:00:00-04:00,288.00,295.00,282.500,292.50,18572,PMI,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
46268,2026-09-22 00:00:00-04:00,2.65,2.95,2.650,2.80,40400,INLF,12.137702,0.001074,-0.004031
46269,2026-09-23 00:00:00-04:00,2.86,2.91,2.834,2.91,16300,INLF,12.083137,0.001084,-0.004048
46270,2026-09-24 00:00:00-04:00,2.90,3.00,2.890,2.95,9533500,INLF,11.880555,0.001147,-0.004112
46271,2026-09-25 00:00:00-04:00,6.25,6.87,4.570,5.10,49411900,INLF,12.155549,0.001953,-0.003953


In [32]:
#date_filt = main_df["Date"] < ("2026-05-01")

In [64]:
date_filt = main_df["Date"] == ("2026-09-11")

In [65]:
stocks_main_filtered = main_df[date_filt]

In [67]:
stocks_main_filtered.describe()

,Open,High,Low,Close,Volume,volatility,growth_252d,Sharpe
count,132.000000,132.000000,132.000000,132.000000,1.320000e+02,132.000000,132.000000,132.000000
mean,12.831901,13.181377,12.510782,12.800431,1.248240e+06,28.116148,1.075491,inf
std,19.213619,19.564220,18.868847,19.135136,3.184362e+06,66.894326,3.041472,NaN
min,0.116000,0.129000,0.115000,0.125000,0.000000e+00,0.000000,0.001005,-0.040147
25%,2.137500,2.257500,2.075500,2.098750,2.297500e+04,2.338723,0.133973,0.011303
50%,4.505000,4.790000,4.475000,4.565000,1.708500e+05,8.164474,0.594534,0.050142
75%,17.397501,17.471250,16.732501,17.224999,9.976250e+05,23.942749,1.045655,0.132363
max,120.500000,120.500000,120.425003,120.425003,2.168250e+07,627.698106,33.638270,inf


In [69]:
stocks_df = main_df.copy()

In [70]:
stocks_df

,Date,Open,High,Low,Close,Volume,Ticker,volatility,growth_252d,Sharpe
0,2025-08-29 00:00:00-04:00,210.50,356.00,209.500,229.50,126794,PMI,NaN,NaN,NaN
1,2025-09-02 00:00:00-04:00,242.50,267.50,232.500,257.00,98358,PMI,NaN,NaN,NaN
2,2025-09-03 00:00:00-04:00,257.00,270.00,250.500,264.50,35960,PMI,NaN,NaN,NaN
3,2025-09-04 00:00:00-04:00,265.50,287.50,265.000,286.50,28074,PMI,NaN,NaN,NaN
4,2025-09-05 00:00:00-04:00,288.00,295.00,282.500,292.50,18572,PMI,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
46268,2026-09-22 00:00:00-04:00,2.65,2.95,2.650,2.80,40400,INLF,12.137702,0.001074,-0.004031
46269,2026-09-23 00:00:00-04:00,2.86,2.91,2.834,2.91,16300,INLF,12.083137,0.001084,-0.004048
46270,2026-09-24 00:00:00-04:00,2.90,3.00,2.890,2.95,9533500,INLF,11.880555,0.001147,-0.004112
46271,2026-09-25 00:00:00-04:00,6.25,6.87,4.570,5.10,49411900,INLF,12.155549,0.001953,-0.003953


In [71]:
stocks_df["future_growth_1_m"] = (stocks_df['Close'].shift(-21) / stocks_df['Close'])  

In [72]:
stocks_df["future_growth_2_m"] = (stocks_df['Close'].shift(-42) / stocks_df['Close'])  

In [73]:
stocks_df["future_growth_3_m"] = (stocks_df['Close'].shift(-63) / stocks_df['Close'])  

In [74]:
stocks_df["future_growth_4_m"] = (stocks_df['Close'].shift(-84) / stocks_df['Close'])  

In [75]:
stocks_df["future_growth_5_m"] = (stocks_df['Close'].shift(-105) / stocks_df['Close'])  

In [76]:
stocks_df["future_growth_6_m"] = (stocks_df['Close'].shift(-126) / stocks_df['Close'])  

In [77]:
stocks_df["future_growth_7_m"] = (stocks_df['Close'].shift(-147) / stocks_df['Close'])  

In [78]:
stocks_df["future_growth_8_m"] = (stocks_df['Close'].shift(-168) / stocks_df['Close'])  

In [79]:
stocks_df["future_growth_9_m"] = (stocks_df['Close'].shift(-189) / stocks_df['Close'])  

In [80]:
stocks_df["future_growth_10_m"] = (stocks_df['Close'].shift(-210) / stocks_df['Close'])  

In [81]:
stocks_df["future_growth_11_m"] = (stocks_df['Close'].shift(-231) / stocks_df['Close'])  

In [82]:
stocks_df["future_growth_12_m"] = (stocks_df['Close'].shift(-252) / stocks_df['Close'])  

In [83]:
stocks_df2 = stocks_df.copy()

In [90]:
pd.set_option("display.max_rows", 150)



In [91]:
stocks_df2

,Date,Open,High,Low,Close,Volume,Ticker,volatility,growth_252d,Sharpe,...,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
0,2025-08-29 00:00:00-04:00,210.50,356.00,209.500,229.50,126794,PMI,NaN,NaN,NaN,...,0.464052,0.381264,0.381264,0.209150,0.228758,0.095643,0.058824,0.036819,0.022353,0.016819
1,2025-09-02 00:00:00-04:00,242.50,267.50,232.500,257.00,98358,PMI,NaN,NaN,NaN,...,0.412451,0.340467,0.402724,0.221790,0.185019,0.068482,0.059922,0.034047,0.016732,0.017821
2,2025-09-03 00:00:00-04:00,257.00,270.00,250.500,264.50,35960,PMI,NaN,NaN,NaN,...,0.374291,0.344045,0.362949,0.202268,0.181474,0.039698,0.060491,0.031758,0.013724,0.016975
3,2025-09-04 00:00:00-04:00,265.50,287.50,265.000,286.50,28074,PMI,NaN,NaN,NaN,...,0.352531,0.291449,0.310646,0.188482,0.181501,0.040489,0.045026,0.026178,0.012356,0.015916
4,2025-09-05 00:00:00-04:00,288.00,295.00,282.500,292.50,18572,PMI,NaN,NaN,NaN,...,0.630769,0.295726,0.288889,0.198291,0.174359,0.037778,0.045128,0.028718,0.011726,0.014735
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46268,2026-09-22 00:00:00-04:00,2.65,2.95,2.650,2.80,40400,INLF,12.137702,0.001074,-0.004031,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46269,2026-09-23 00:00:00-04:00,2.86,2.91,2.834,2.91,16300,INLF,12.083137,0.001084,-0.004048,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46270,2026-09-24 00:00:00-04:00,2.90,3.00,2.890,2.95,9533500,INLF,11.880555,0.001147,-0.004112,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46271,2026-09-25 00:00:00-04:00,6.25,6.87,4.570,5.10,49411900,INLF,12.155549,0.001953,-0.003953,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [97]:
first_dates3 = (
    stocks_df2
    .dropna(subset=["Close"])
    .sort_values(["Ticker", "Date"])
    .groupby("Ticker", as_index=False)
    .agg(
        first_date=("Date", "first"),
        first_close=("Close", "first")
        
    )
)

In [98]:
first_dates3

,Ticker,first_date,first_close
0,AAPG,2025-01-24 00:00:00-05:00,17.379999
1,AARD,2025-02-13 00:00:00-05:00,14.310000
2,ADVB,2025-03-06 00:00:00-05:00,73.000000
3,AII,2025-05-08 00:00:00-04:00,16.008226
4,AIRO,2025-06-13 00:00:00-04:00,24.000000
5,ALM,2025-01-02 00:00:00-05:00,1.552500
6,AMBQ,2025-07-30 00:00:00-04:00,38.529999
7,ANPA,2025-07-08 00:00:00-04:00,3.520000
8,ANTA,2025-05-13 00:00:00-04:00,12.800000
9,APUS,2025-05-09 00:00:00-04:00,22.000000


In [101]:
first_dates3.rename(
    columns={"first_date": "Date"}, inplace=True
)

In [107]:
first_dates3.shape

(132, 3)

In [103]:
min_date_close = first_dates3.copy()

In [104]:
merged_df = min_date_close.merge(
    stocks_df2,
    on=["Ticker", "Date"],
    how="inner"
)

merged_df.shape

 stocks_df2.shape

In [111]:
merged_df.describe()

,first_close,Open,High,Low,Close,Volume,volatility,growth_252d,Sharpe,future_growth_1_m,...,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
count,132.000000,132.000000,132.000000,132.000000,132.000000,1.320000e+02,130.000000,131.000000,130.000000,132.000000,...,132.000000,132.000000,132.000000,132.000000,132.000000,132.000000,132.000000,132.000000,132.000000,132.000000
mean,195.023703,180.660083,226.635446,161.229042,195.023703,6.053674e+06,573.450369,78.449049,0.070354,95.751832,...,50.899990,22.555451,53.856962,63.548719,58.842056,21.334088,19.887079,11.199384,8.863178,5.780039
std,1293.714942,1094.391668,1460.418167,1030.146852,1293.714942,1.113548e+07,3770.784656,683.332094,0.109189,1087.893336,...,571.091208,246.883476,607.008003,717.979917,663.580095,233.839993,215.341507,115.274013,89.691458,55.072876
min,0.020000,0.015000,0.020000,0.010000,0.020000,0.000000e+00,2.913601,0.004608,-0.005899,0.072941,...,0.090147,0.063927,0.032245,0.022075,0.013229,0.015130,0.014675,0.011947,0.008681,0.004833
25%,4.577500,4.587793,5.712500,4.000000,4.577500,2.408400e+05,20.553565,0.344076,0.006140,0.710433,...,0.443759,0.398507,0.419447,0.310688,0.249697,0.197711,0.176164,0.177117,0.164555,0.162199
50%,17.812232,18.627110,20.875260,16.119598,17.812232,1.696600e+06,51.047462,1.054545,0.026499,0.935351,...,0.827952,0.736039,0.695853,0.733385,0.667972,0.603984,0.584421,0.533487,0.487132,0.529827
75%,37.365000,39.625000,42.689999,36.204999,37.365000,6.050525e+06,117.280947,4.351511,0.075201,1.114199,...,1.149382,1.200740,1.110982,1.056055,1.019214,1.038017,1.040995,1.046610,1.142106,1.081503
max,14240.000000,11840.000000,16160.000000,11232.000000,14240.000000,6.440900e+07,41266.860344,7739.130294,0.612708,12500.000279,...,6562.500147,2837.500063,6975.000156,8250.000184,7625.000170,2687.500060,2475.000055,1325.000030,1031.250023,633.250012


In [114]:
return_columns = [
    f"future_growth_{month}_m"
    for month in range(1, 13)
]

In [115]:
return_statistics = merged_df[
    return_columns
].describe()



In [116]:
return_statistics

,future_growth_1_m,future_growth_2_m,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
count,132.000000,132.000000,132.000000,132.000000,132.000000,132.000000,132.000000,132.000000,132.000000,132.000000,132.000000,132.000000
mean,95.751832,67.502036,50.899990,22.555451,53.856962,63.548719,58.842056,21.334088,19.887079,11.199384,8.863178,5.780039
std,1087.893336,761.485301,571.091208,246.883476,607.008003,717.979917,663.580095,233.839993,215.341507,115.274013,89.691458,55.072876
min,0.072941,0.106197,0.090147,0.063927,0.032245,0.022075,0.013229,0.015130,0.014675,0.011947,0.008681,0.004833
25%,0.710433,0.587322,0.443759,0.398507,0.419447,0.310688,0.249697,0.197711,0.176164,0.177117,0.164555,0.162199
50%,0.935351,0.893217,0.827952,0.736039,0.695853,0.733385,0.667972,0.603984,0.584421,0.533487,0.487132,0.529827
75%,1.114199,1.188819,1.149382,1.200740,1.110982,1.056055,1.019214,1.038017,1.040995,1.046610,1.142106,1.081503
max,12500.000279,8750.000196,6562.500147,2837.500063,6975.000156,8250.000184,7625.000170,2687.500060,2475.000055,1325.000030,1031.250023,633.250012


In [117]:
median_returns = return_statistics.loc["50%"]

In [123]:
mean_returns = return_statistics.loc["mean"]

In [124]:
mean_returns

future_growth_1_m     95.751832
future_growth_2_m     67.502036
future_growth_3_m     50.899990
future_growth_4_m     22.555451
future_growth_5_m     53.856962
future_growth_6_m     63.548719
future_growth_7_m     58.842056
future_growth_8_m     21.334088
future_growth_9_m     19.887079
future_growth_10_m    11.199384
future_growth_11_m     8.863178
future_growth_12_m     5.780039
Name: mean, dtype: float64

In [120]:
median_returns

future_growth_1_m     0.935351
future_growth_2_m     0.893217
future_growth_3_m     0.827952
future_growth_4_m     0.736039
future_growth_5_m     0.695853
future_growth_6_m     0.733385
future_growth_7_m     0.667972
future_growth_8_m     0.603984
future_growth_9_m     0.584421
future_growth_10_m    0.533487
future_growth_11_m    0.487132
future_growth_12_m    0.529827
Name: 50%, dtype: float64

In [121]:
median_returns.idxmax()

'future_growth_1_m'

In [122]:
median_returns.max()

0.9353509333873746